# Four trust-region reruns

This notebook loads the last NN-solver iterate from results/000, results/075, results/150, and results/rng42, then runs the current trust-region configuration once for each warm start. Each TR result is written directly back to its matching existing directory; timestamped filenames preserve earlier runs.

In [1]:
import gc
import time
import traceback
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import pandapower.networks as pn

from stackelberg.followers.bg import create_cigre_residential_network
from stackelberg.followers.ip import create_industrial_park_network
from stackelberg.network import create_case33bw_20kv
from stackelberg.trust_region import (
    TrustRegionExperimentConfig,
    run_trust_region_experiment,
)

assert Path("stackelberg").is_dir(), "Run this notebook from the project root."


## Shared experiment settings

RUN_BATCH=True makes **Run All** execute all five experiments. CONTINUE_ON_ERROR=True records a failed run and proceeds to the next warm start. The initial global solve uses zero Adam steps; COPT network-state initialization is enabled only for the local trust-region solves.

In [2]:
RUN_BATCH = True
CONTINUE_ON_ERROR = True

EXPERIMENT_LABELS = ("000", "075", "150", "rng42")
RESULTS_ROOT = Path("results")
T = 24
SEED = 42
SCENARIOS = ("SB1", "SB2", "BG1", "BG2", "IP1")
MG_BUSES = (17, 32, 15, 21, 1)
EXPECTED_PRICE_COUNT = len(SCENARIOS) * T

# These values mirror the current run_trust_region_experiment.ipynb setup.
TR_PARAMS = {
    "R_0": 0.05,
    "R_max": 1.0,
    "gamma_dec": 0.7,
    "gamma_inc": 1.5,
    "eta_1": 0.1,
    "eta_2": 0.75,
}
SOLVE_PARAMS = {
    "steps_init": 0,
    "steps_warm": 1000,
    "lr": 1e-2,
    "penalty_weights": {
        "eq": 1000.0,
        "ineq": 100.0,
        "soc": 100.0,
        "bound": 10.0,
    },
    "tr_weight": 1e6,
    "verbose": True,
}
ALGO_PARAMS = {
    "n_samples": 60,
    "K_max": 60,
    "sampling_method": "lhs",
    "residual_update_mode": "accumulate",
    "rbf_geometry_condition_max": 1e12,
    "solve_params": SOLVE_PARAMS,
}
RESIDUAL_MODEL_PARAMS = {
    "model_type": "rbf",
    "rbf_kernel": "cubic",
    "rbf_smoothing": 1e-3,
    "rbf_degree": 1,
    "rbf_scale": "trust_radius",
    "rbf_epsilon": None,
}


In [3]:
def build_networks():
    """Return fresh network objects so runs cannot leak mutable state."""
    upper_net = create_case33bw_20kv()
    lower_nets = [
        pn.create_cigre_network_lv(),
        pn.create_cigre_network_lv(),
        create_cigre_residential_network(),
        create_cigre_residential_network(),
        create_industrial_park_network(),
    ]
    return upper_net, lower_nets


def latest_nn_history(results_dir: Path) -> Path:
    candidates = list(results_dir.glob("NN_solver_history_*.csv"))
    fallback = results_dir / "NN_solver_history.csv"
    if not candidates and fallback.is_file():
        candidates = [fallback]
    if not candidates:
        raise FileNotFoundError(
            f"No NN solver history found in {results_dir}."
        )
    return max(candidates, key=lambda path: path.stat().st_mtime_ns)


def numbered_price_columns(frame: pd.DataFrame) -> list[str]:
    indexed_columns = []
    for column in frame.columns:
        if not column.startswith("pi_"):
            continue
        try:
            index = int(column.removeprefix("pi_"))
        except ValueError:
            continue
        indexed_columns.append((index, column))
    indexed_columns.sort()
    indices = [index for index, _ in indexed_columns]
    expected = list(range(EXPECTED_PRICE_COUNT))
    if indices != expected:
        raise ValueError(
            "NN history price columns must be exactly pi_0 through "
            f"pi_{EXPECTED_PRICE_COUNT - 1}; found {indices}."
        )
    return [column for _, column in indexed_columns]


def load_nn_warm_start(label: str):
    results_dir = RESULTS_ROOT / label
    history_path = latest_nn_history(results_dir)
    history = pd.read_csv(history_path)
    if history.empty:
        raise ValueError(f"NN solver history is empty: {history_path}")
    pi_columns = numbered_price_columns(history)
    ordered = history.sort_values("iteration") if "iteration" in history else history
    last_row = ordered.iloc[-1]
    warm_start = last_row[pi_columns].to_numpy(dtype=float).reshape(
        len(SCENARIOS), T
    )
    if not np.isfinite(warm_start).all():
        raise ValueError(f"Warm start contains non-finite values: {history_path}")
    metadata = {
        "label": label,
        "history_path": str(history_path),
        "iteration": int(last_row["iteration"])
        if "iteration" in last_row.index
        else len(history) - 1,
        "nn_true_loss": float(last_row["true_loss"])
        if "true_loss" in last_row.index
        else np.nan,
        "price_min": float(warm_start.min()),
        "price_max": float(warm_start.max()),
    }
    return warm_start, metadata


## Warm-start preflight

All five inputs are loaded and validated before any trust-region run begins. Check the displayed file paths before starting the batch.

In [4]:
warm_starts = {}
warm_start_records = []
for label in EXPERIMENT_LABELS:
    warm_start, metadata = load_nn_warm_start(label)
    assert warm_start.shape == (len(SCENARIOS), T)
    warm_starts[label] = warm_start
    warm_start_records.append(metadata)

warm_start_summary = pd.DataFrame(warm_start_records)
display(warm_start_summary)

duplicate_pairs = [
    (left, right)
    for index, left in enumerate(EXPERIMENT_LABELS)
    for right in EXPERIMENT_LABELS[index + 1:]
    if np.array_equal(warm_starts[left], warm_starts[right])
]
if duplicate_pairs:
    print("WARNING: identical NN warm starts found:", duplicate_pairs)


,label,history_path,iteration,nn_true_loss,price_min,price_max
0,000,results/000/NN_solver_history_20260912-234954.csv,9800,-103.737458,0.261827,1.499971
1,075,results/075/NN_solver_history_20260911-215335.csv,9800,-98.432021,0.947484,1.499947
2,150,results/150/NN_solver_history_20260913-151813.csv,9800,-93.641439,0.952476,1.500000
3,rng42,results/rng42/NN_solver_history_20260914-12015...,9800,-97.986705,0.950017,1.499993


In [5]:
def make_tr_config(label: str, warm_start: np.ndarray):
    upper_net, lower_nets = build_networks()
    return TrustRegionExperimentConfig(
        upper_net=upper_net,
        lower_nets=lower_nets,
        mg_buses=MG_BUSES,
        scenarios=SCENARIOS,
        T=T,
        seed=SEED,
        device="cpu",
        x0=warm_start,
        lower_time_limit=30.0,
        lower_mip_gap=1e-3,
        copt_initialize_network_state=True,
        tr_params=dict(TR_PARAMS),
        algo_params={
            **ALGO_PARAMS,
            "solve_params": {
                **SOLVE_PARAMS,
                "penalty_weights": dict(SOLVE_PARAMS["penalty_weights"]),
            },
        },
        residual_model_params=dict(RESIDUAL_MODEL_PARAMS),
        save_results=True,
        output_dir=str(RESULTS_ROOT / label),
        save_true_responses=False,
    )


# Lightweight configuration check; this does not build a controller or call COPT.
for label in EXPERIMENT_LABELS:
    config = make_tr_config(label, warm_starts[label])
    assert config.copt_initialize_network_state is True
    assert config.algo_params["solve_params"]["steps_init"] == 0
    assert config.algo_params["solve_params"]["steps_warm"] == 1000
    assert Path(config.output_dir) == RESULTS_ROOT / label
del config


## Run all five experiments

The runs are sequential to avoid competing COPT/PyTorch workloads. The summary CSV is rewritten after every completed or failed run, while the full timestamped TR JSON/CSV files are produced inside the matching result directory.

In [6]:
batch_id = datetime.now().strftime("%Y%m%d-%H%M%S")
batch_summary_path = RESULTS_ROOT / f"trust_region_five_runs_{batch_id}.csv"
batch_records = []

if RUN_BATCH:
    for run_number, label in enumerate(EXPERIMENT_LABELS, start=1):
        print("\n" + "=" * 80)
        print(f"[{run_number}/{len(EXPERIMENT_LABELS)}] Starting TR run: {label}")
        print("Warm start:", warm_start_summary.loc[
            warm_start_summary["label"] == label, "history_path"
        ].iloc[0])
        print("Output directory:", RESULTS_ROOT / label)
        started = time.perf_counter()
        record = {
            "label": label,
            "status": "running",
            "warm_start_history": warm_start_summary.loc[
                warm_start_summary["label"] == label, "history_path"
            ].iloc[0],
            "output_dir": str(RESULTS_ROOT / label),
        }
        config = None
        result = None
        try:
            config = make_tr_config(label, warm_starts[label])
            result = run_trust_region_experiment(config)
            last_history = result["history"][-1] if result["history"] else {}
            saved_paths = result.get("saved_paths", {})
            record.update(
                status="complete",
                elapsed_seconds=time.perf_counter() - started,
                n_iter=int(result["n_iter"]),
                converged=bool(result["converged"]),
                stop_reason=result.get("stop_reason"),
                final_true_objective=last_history.get("F_next", np.nan),
                result_json=saved_paths.get("json"),
                history_csv=saved_paths.get("csv"),
                error=None,
            )
            print(f"Completed {label} in {record['elapsed_seconds']:.1f} seconds.")
            print("Saved:", saved_paths)
        except Exception as exc:
            record.update(
                status="failed",
                elapsed_seconds=time.perf_counter() - started,
                error=f"{type(exc).__name__}: {exc}",
            )
            traceback.print_exc()
        finally:
            batch_records.append(record)
            pd.DataFrame(batch_records).to_csv(batch_summary_path, index=False)
            del result, config
            gc.collect()

        if record["status"] == "failed" and not CONTINUE_ON_ERROR:
            raise RuntimeError(
                f"Batch stopped after {label}: {record['error']}"
            )
else:
    print("RUN_BATCH=False: preflight completed; no TR experiment was run.")

batch_summary = pd.DataFrame(batch_records)
print("Batch summary:", batch_summary_path)
display(batch_summary)



[1/4] Starting TR run: 000
Warm start: results/000/NN_solver_history_20260912-234954.csv
Output directory: results/000
Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Using global surrogate to solve initial point...
Initial surrogate solve used steps_init=0. Time taken to solve initial point 0.0485 seconds.
Iteration 0 | R=0.0500 
Building 60-point local designs for each of 5 followers (center + coordi

,label,status,warm_start_history,output_dir,elapsed_seconds,n_iter,converged,stop_reason,final_true_objective,result_json,history_csv,error
0,000,complete,results/000/NN_solver_history_20260912-234954.csv,results/000,8801.796164,31,True,converged,-105.757233,results/000/trust_region_result_20260924-23445...,results/000/trust_region_history_20260924-2344...,None
1,075,complete,results/075/NN_solver_history_20260911-215335.csv,results/075,14807.607400,52,True,converged,-100.717135,results/075/trust_region_result_20260925-03514...,results/075/trust_region_history_20260925-0351...,None
2,150,complete,results/150/NN_solver_history_20260913-151813.csv,results/150,5296.830953,20,True,converged,-94.632933,results/150/trust_region_result_20260925-05200...,results/150/trust_region_history_20260925-0520...,None
3,rng42,complete,results/rng42/NN_solver_history_20260914-12015...,results/rng42,10258.011713,37,True,converged,-100.034189,results/rng42/trust_region_result_20260925-081...,results/rng42/trust_region_history_20260925-08...,None
